# Schedule Comparison: Proposition 4 (Zeta-2) vs Proposition 5 (Stitched)

Compares the two anytime composition schedules for K_k (prepaid compositions after k hits):

| Schedule | Overhead | Reference |
|---|---|---|
| Zeta-2 | Θ(√(k·log(k/δ))/q) | Proposition 4 |
| Stitched | Θ(√((1-q)·k·log log k)/q) | Proposition 5 |

**Key distinction:** The stitched schedule unions failure budget over epochs j = ⌈log₂(k∨2)⌉ rather than individual hit counts. This replaces log(k) with log(log k) in the asymptotic overhead — sharper by Proposition 6 (minimax optimal).

**Practical caveat:** The stitched formula has a large *additive* term 8β_j/q from the Ville's inequality step. For small k this dominates, making stitched *worse* than Zeta-2. The crossover where stitched < Zeta-2 depends on parameters; for the paper defaults (ε=0.01, δ=1e-2) it occurs near k≈150 hits. The stitched schedule is most beneficial for users who accumulate many target hits.


In [ ]:
import sys
sys.path.insert(0, '..')

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from simulation.accountants.adaptive_tct import AdaptiveTCTAccountant, _C_PRIME

plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 10,
    'axes.labelsize': 10,
    'axes.titlesize': 11,
    'legend.fontsize': 9,
    'lines.linewidth': 2.0,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

ZETA_COLOR    = '#2166ac'
STITCH_COLOR  = '#1a9641'
BASELINE_COLOR = '#888888'

EPSILON = 0.01
DELTA   = 1e-2

acc_zeta2    = AdaptiveTCTAccountant(EPSILON, DELTA, schedule='zeta2')
acc_stitched = AdaptiveTCTAccountant(EPSILON, DELTA, schedule='stitched')
q = acc_zeta2.q

print(f'ε={EPSILON}, δ={DELTA}, q={q:.4f}')
print(f'c\' = {_C_PRIME:.6f}  (Proposition 5 constant)')

## Section 1: K_k Values and Overhead

Direct comparison up to k=200. Note the crossover: stitched starts higher (large additive term)
but crosses below Zeta-2 around k≈150. Negative saving% = stitched is worse there.


In [ ]:
k_range = list(range(1, 201))

K_zeta2    = [acc_zeta2.compositions_zeta2(k)    for k in k_range]
K_stitched = [acc_stitched.compositions_stitched(k) for k in k_range]
K_baseline = [k / q for k in k_range]   # k/q: baseline with no overhead

overhead_zeta2    = [K_zeta2[i] - K_baseline[i]    for i in range(len(k_range))]
overhead_stitched = [K_stitched[i] - K_baseline[i] for i in range(len(k_range))]
saving = [overhead_zeta2[i] - overhead_stitched[i] for i in range(len(k_range))]
saving_pct = [100 * saving[i] / max(overhead_zeta2[i], 1e-9) for i in range(len(k_range))]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Panel 1: absolute K_k
ax = axes[0]
ax.plot(k_range, K_baseline, color=BASELINE_COLOR, lw=1.2, linestyle=':', label=r'Baseline $k/q$')
ax.plot(k_range, K_zeta2,    color=ZETA_COLOR,    lw=2, label='Zeta-2 (Prop 4)')
ax.plot(k_range, K_stitched, color=STITCH_COLOR,  lw=2, linestyle='--', label='Stitched (Prop 5)')
ax.set_xlabel('Target hits k')
ax.set_ylabel(r'$K_k$ (prepaid compositions)')
ax.set_title('Prepaid composition count $K_k$')
ax.legend()

# Panel 2: overhead K_k - k/q
ax = axes[1]
ax.plot(k_range, overhead_zeta2,    color=ZETA_COLOR,   lw=2, label='Zeta-2 overhead')
ax.plot(k_range, overhead_stitched, color=STITCH_COLOR, lw=2, linestyle='--', label='Stitched overhead')
ax.fill_between(k_range, overhead_stitched, overhead_zeta2,
                alpha=0.15, color=STITCH_COLOR, label='Stitched saving')
ax.set_xlabel('Target hits k')
ax.set_ylabel(r'$K_k - k/q$  (anytime overhead)')
ax.set_title('Anytime overhead')
ax.legend()

# Panel 3: saving as % of Zeta-2 overhead
ax = axes[2]
ax.plot(k_range, saving_pct, color=STITCH_COLOR, lw=2)
ax.axhline(0, color=BASELINE_COLOR, lw=0.8, linestyle=':')
ax.set_xlabel('Target hits k')
ax.set_ylabel('Overhead saving (% of Zeta-2)')
ax.set_title('Stitched overhead saving vs Zeta-2')

fig.suptitle(
    rf'Schedule Comparison — $\varepsilon$={EPSILON}, $\delta$={DELTA}, $q$={q:.4f}',
    fontsize=10,
)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig('../figures/schedule_comparison_overhead.png', bbox_inches='tight', dpi=200)
plt.show()

# Table
rows = []
for k in [1, 2, 5, 10, 20, 50, 100, 200]:
    i = k - 1
    rows.append({
        'k': k,
        'k/q': f'{K_baseline[i]:.1f}',
        'K_k (Zeta-2)': f'{K_zeta2[i]:.1f}',
        'K_k (Stitched)': f'{K_stitched[i]:.1f}',
        'Saving': f'{saving[i]:.1f}',
        'Saving %': f'{saving_pct[i]:.1f}%',
    })
print(pd.DataFrame(rows).to_string(index=False))

## Section 2: Epoch Structure of the Stitched Schedule

The stitched schedule is piecewise: within epoch j = ⌈log₂(k∨2)⌉, β_j is constant.
This creates step-function behavior in the overhead — visible as plateaus in the curve.

In [ ]:
# Show epoch boundaries and β_j values
print('Epoch structure:')
print(f'{"j":>4}  {"k range":>12}  {"δ_j":>12}  {"β_j":>8}')
print('-' * 45)
for j in range(1, 9):
    k_lo = 2**(j-1) + 1 if j > 1 else 1
    k_hi = 2**j
    delta_j = 6 / (math.pi**2 * j**2) * DELTA
    beta_j  = math.log(math.pi**2 * j**2 / (6 * DELTA))
    print(f'{j:>4}  {k_lo:>5} – {k_hi:<5}  {delta_j:>12.2e}  {beta_j:>8.4f}')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# β values side-by-side
beta_zeta2    = [math.log(math.pi**2 * k**2 / (6 * DELTA))  for k in k_range]
beta_stitched = [
    math.log(math.pi**2 * math.ceil(math.log2(max(k,2)))**2 / (6 * DELTA))
    for k in k_range
]

ax = axes[0]
ax.plot(k_range, beta_zeta2,    color=ZETA_COLOR,   lw=2, label=r'$\beta_k$ (Zeta-2): $\log(\pi^2 k^2/6\delta)$')
ax.plot(k_range, beta_stitched, color=STITCH_COLOR, lw=2, linestyle='--',
        label=r'$\beta_{j(k)}$ (Stitched): $\log(\pi^2 j^2/6\delta)$')
ax.set_xlabel('Target hits k')
ax.set_ylabel(r'$\beta$ value')
ax.set_title(r'$\beta$ used in schedule')
ax.legend(fontsize=8)

# j(k) function
j_vals = [math.ceil(math.log2(max(k, 2))) for k in k_range]
ax = axes[1]
ax.step(k_range, j_vals, color=STITCH_COLOR, lw=2, where='post')
for j in range(1, 9):
    ax.axvline(2**j, color=BASELINE_COLOR, lw=0.7, linestyle=':', alpha=0.7)
ax.set_xlabel('Target hits k')
ax.set_ylabel(r'Epoch $j(k) = \lceil \log_2(k \vee 2) \rceil$')
ax.set_title('Epoch index j(k)')

fig.suptitle(f'Stitched schedule epoch structure  (δ={DELTA:.0e})', fontsize=10)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig('../figures/schedule_comparison_epochs.png', bbox_inches='tight', dpi=200)
plt.show()

## Section 3: Asymptotic Scaling

Verify the theoretical overhead rates:
- Zeta-2: overhead / √(k·log k) → constant
- Stitched: overhead / √(k·log log k) → constant

In [ ]:
k_large = list(range(10, 5001, 10))

K_z_lg = [acc_zeta2.compositions_zeta2(k)       for k in k_large]
K_s_lg = [acc_stitched.compositions_stitched(k)  for k in k_large]
K_b_lg = [k / q for k in k_large]

oh_z = [K_z_lg[i] - K_b_lg[i] for i in range(len(k_large))]
oh_s = [K_s_lg[i] - K_b_lg[i] for i in range(len(k_large))]

# Normalize by theoretical scaling
norm_z = [oh_z[i] / (math.sqrt(k_large[i] * math.log(k_large[i])) / q)
          for i in range(len(k_large))]
norm_s = [oh_s[i] / (math.sqrt(k_large[i] * math.log(math.log(max(k_large[i], 3)))) / q)
          for i in range(len(k_large))]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
ax.loglog(k_large, oh_z, color=ZETA_COLOR,   lw=2, label='Zeta-2')
ax.loglog(k_large, oh_s, color=STITCH_COLOR, lw=2, linestyle='--', label='Stitched')
# Reference slopes
ref_k = k_large[-1]
ref_z = oh_z[-1]
ax.loglog(k_large, [ref_z * math.sqrt(k * math.log(k)) / math.sqrt(ref_k * math.log(ref_k))
                    for k in k_large],
          color=ZETA_COLOR, lw=0.8, linestyle=':', alpha=0.6, label=r'$\propto\sqrt{k\log k}$')
ax.loglog(k_large, [ref_z * math.sqrt(k * math.log(math.log(max(k,3)))) /
                    math.sqrt(ref_k * math.log(math.log(ref_k)))
                    for k in k_large],
          color=STITCH_COLOR, lw=0.8, linestyle=':', alpha=0.6, label=r'$\propto\sqrt{k\log\log k}$')
ax.set_xlabel('Target hits k')
ax.set_ylabel('Overhead $K_k - k/q$')
ax.set_title('Overhead (log-log)')
ax.legend(fontsize=8)

ax = axes[1]
ax.semilogx(k_large, norm_z, color=ZETA_COLOR,   lw=2, label=r'Zeta-2 / $\sqrt{k\log k}/q$')
ax.semilogx(k_large, norm_s, color=STITCH_COLOR, lw=2, linestyle='--',
            label=r'Stitched / $\sqrt{k\log\log k}/q$')
ax.axhline(0, color=BASELINE_COLOR, lw=0.5, linestyle=':')
ax.set_xlabel('Target hits k')
ax.set_ylabel('Normalized overhead')
ax.set_title('Overhead / theoretical rate\n(converges to constant if rate is tight)')
ax.legend(fontsize=8)

fig.suptitle(
    rf'Asymptotic overhead scaling — $\varepsilon$={EPSILON}, $\delta$={DELTA}',
    fontsize=10,
)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig('../figures/schedule_comparison_scaling.png', bbox_inches='tight', dpi=200)
plt.show()

## Section 4: Privacy Cost Impact (RDP mode)

Under RDP accounting (ρ = α·ε²/2), the privacy cost is K_k·ρ.
How much budget does each schedule consume as a function of hits k?

In [ ]:
ALPHA_RDP  = 100.0
rho        = ALPHA_RDP * EPSILON**2 / 2
EPS_BUDGET = 1.0

acc_z_rdp = AdaptiveTCTAccountant(EPSILON, DELTA, alpha_rdp=ALPHA_RDP, schedule='zeta2')
acc_s_rdp = AdaptiveTCTAccountant(EPSILON, DELTA, alpha_rdp=ALPHA_RDP, schedule='stitched')

k_range_rdp = list(range(1, 101))
cost_z   = [acc_z_rdp.budget_cost(acc_z_rdp.compositions_zeta2(k))       for k in k_range_rdp]
cost_s   = [acc_s_rdp.budget_cost(acc_s_rdp.compositions_stitched(k))     for k in k_range_rdp]
cost_idp = [k * rho for k in k_range_rdp]   # standard IDP: k·ρ

# Budget exhaustion (max k where cost ≤ budget)
def max_hits(costs, budget):
    for k, c in enumerate(costs, start=1):
        if c > budget:
            return k - 1
    return len(costs)

k_exhaust_z   = max_hits(cost_z,   EPS_BUDGET)
k_exhaust_s   = max_hits(cost_s,   EPS_BUDGET)
k_exhaust_idp = max_hits(cost_idp, EPS_BUDGET)

print(f'Budget = {EPS_BUDGET},  ρ = {rho:.5f},  α = {ALPHA_RDP}')
print(f'Standard IDP exhausted after {k_exhaust_idp} hits')
print(f'Zeta-2  TCT exhausted after  {k_exhaust_z} hits')
print(f'Stitched TCT exhausted after {k_exhaust_s} hits')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
ax.plot(k_range_rdp, cost_idp, color='#d6604d', lw=2, linestyle=':',  label=f'Std IDP RDP  (exhausts k={k_exhaust_idp})')
ax.plot(k_range_rdp, cost_z,   color=ZETA_COLOR,   lw=2,              label=f'Zeta-2 TCT   (exhausts k={k_exhaust_z})')
ax.plot(k_range_rdp, cost_s,   color=STITCH_COLOR, lw=2, linestyle='--', label=f'Stitched TCT (exhausts k={k_exhaust_s})')
ax.axhline(EPS_BUDGET, color='black', lw=0.9, linestyle=':', label=rf'$\varepsilon_{{RDP}}$ budget = {EPS_BUDGET}')
ax.set_xlabel('Target hits k')
ax.set_ylabel(r'Cumulative $\varepsilon_{\mathrm{RDP}}$ charged')
ax.set_title('RDP privacy cost vs hits')
ax.legend(fontsize=8)

ax = axes[1]
ratio_zs  = [cost_z[i] / cost_s[i] for i in range(len(k_range_rdp))]
ratio_idpz = [cost_idp[i] / cost_z[i] for i in range(len(k_range_rdp))]
ax.plot(k_range_rdp, ratio_zs,   color=STITCH_COLOR, lw=2, label='Zeta-2 / Stitched')
ax.plot(k_range_rdp, ratio_idpz, color='#d6604d',    lw=2, linestyle='--', label='Std IDP / Zeta-2')
ax.axhline(1.0, color=BASELINE_COLOR, lw=0.8, linestyle=':')
ax.set_xlabel('Target hits k')
ax.set_ylabel('Cost ratio')
ax.set_title('Privacy cost ratios')
ax.legend(fontsize=9)

fig.suptitle(
    rf'RDP Privacy Cost — $\varepsilon$={EPSILON}, $\alpha$={ALPHA_RDP:.0f}, $\rho$={rho:.5f}, $\delta$={DELTA}',
    fontsize=10,
)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig('../figures/schedule_comparison_rdp_cost.png', bbox_inches='tight', dpi=200)
plt.show()

## Section 5: H1 Simulation — Retention Curve Comparison

Run the full H1 simulation (S1: full participation, no interleaving) under both schedules
to see how the overhead difference translates to user retention.

In [ ]:
import numpy as np
from simulation.accountants.standard_dp import StandardDPAccountant
from simulation.calibration import q_from_epsilon
from simulation.datasets import generate_synthetic_population, PopulationConfig
from simulation.query import compute_round_conversions, laplace_threshold_query

N_USERS        = 5000
N_ROUNDS       = 500
EPS_RDP_BUDGET = 1.0
TARGET_HIT     = 0.02
SEED           = 42

pop_cfg = PopulationConfig(
    n_users=N_USERS, alpha=0.5, beta=2.0,
    epsilon_budget=EPS_RDP_BUDGET, delta_budget=1.0,
)

def calibrate(pop_cfg, seed=SEED):
    rng = np.random.default_rng(seed)
    pop = generate_synthetic_population(pop_cfg, rng=np.random.default_rng(seed))
    noisy = []
    for _ in range(3000):
        parts = list(pop.active_users)
        cnt, _ = compute_round_conversions(parts, rng)
        noisy.append(cnt + rng.laplace(0, 1/EPSILON))
    return float(np.quantile(noisy, 1 - TARGET_HIT))

def run_sim(accountant_factory, pop_cfg, T, seed=SEED):
    rng = np.random.default_rng(seed)
    pop = generate_synthetic_population(pop_cfg, rng=np.random.default_rng(seed))
    acc = accountant_factory()
    active_frac = []
    for r in range(N_ROUNDS):
        active = pop.active_users
        if not active:
            break
        parts = list(active)
        cnt, contribs = compute_round_conversions(parts, rng)
        is_top, _ = laplace_threshold_query(cnt, T, EPSILON, rng)
        for i, user in enumerate(parts):
            e, d = acc.charge(user=user, is_top=is_top,
                              contributed=contribs[i], round_num=r)
            user.epsilon_spent += e
            user.delta_spent   += d
        pop.drop_depleted()
        active_frac.append(pop.active_fraction)
    return active_frac

print('Calibrating threshold...')
T = calibrate(pop_cfg)
print(f'Threshold T = {T:.2f}')

print('Running simulations...')
frac_z = run_sim(lambda: AdaptiveTCTAccountant(EPSILON, DELTA, alpha_rdp=ALPHA_RDP, schedule='zeta2'),    pop_cfg, T)
frac_s = run_sim(lambda: AdaptiveTCTAccountant(EPSILON, DELTA, alpha_rdp=ALPHA_RDP, schedule='stitched'), pop_cfg, T)
frac_i = run_sim(lambda: StandardDPAccountant(rho), pop_cfg, T)

def half_life(frac):
    for r, f in enumerate(frac):
        if f <= 0.5:
            return r
    return len(frac)

hl_z = half_life(frac_z)
hl_s = half_life(frac_s)
hl_i = half_life(frac_i)
print(f'Half-lives:  Zeta-2={hl_z}  Stitched={hl_s}  IDP={hl_i}')

In [ ]:
n = max(len(frac_z), len(frac_s), len(frac_i)) + 1
def pad(c, n):
    return list(c) + [c[-1] if c else 0.0] * (n - len(c))

rnds = list(range(n))
fz = pad(frac_z, n)
fs = pad(frac_s, n)
fi = pad(frac_i, n)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rnds, fi, color='#d6604d', lw=2, linestyle=':', label=f'Std IDP RDP  (hl={hl_i})')
ax.plot(rnds, fz, color=ZETA_COLOR,   lw=2,              label=f'TCT Zeta-2   (hl={"≥"+str(hl_z) if hl_z>=n-2 else hl_z})')
ax.plot(rnds, fs, color=STITCH_COLOR, lw=2, linestyle='--', label=f'TCT Stitched (hl={"≥"+str(hl_s) if hl_s>=n-2 else hl_s})')
ax.fill_between(rnds, fz, fs, alpha=0.15, color=STITCH_COLOR, label='Stitched − Zeta-2 gain')
ax.axhline(0.5, color='#aaa', lw=0.8, linestyle=':', zorder=0)
if hl_i < n - 2:
    ax.axvline(hl_i, color='#d6604d', lw=0.8, linestyle=':', alpha=0.6)
ax.set_xlabel('Query round')
ax.set_ylabel('Fraction of users active')
ax.set_title(
    rf'H1: Schedule comparison — Full participation, no interleaving'
    rf'  ($\varepsilon$={EPSILON}, $\alpha$={ALPHA_RDP:.0f}, $N$={N_USERS})'
)
ax.set_xlim(0, n - 1); ax.set_ylim(-0.02, 1.08)
ax.legend()
plt.tight_layout()
plt.savefig('../figures/schedule_comparison_h1.png', bbox_inches='tight', dpi=200)
plt.show()

## Section 6: Summary

In [ ]:
print('=== Schedule Comparison Summary ===')
print(f'Parameters: ε={EPSILON}, δ={DELTA}, α={ALPHA_RDP}, ρ={rho:.5f}')
print(f'RDP budget per user: {EPS_RDP_BUDGET}')
print()
print(f'{"Schedule":<20} {"K_10":>8} {"K_50":>8} {"K_100":>8} {"Max hits (budget)".ljust(20)}')
print('-' * 62)
print(f'{"Zeta-2 (Prop 4)":<20}'
      f' {acc_z_rdp.budget_cost(acc_z_rdp.compositions_zeta2(10)):>8.3f}'
      f' {acc_z_rdp.budget_cost(acc_z_rdp.compositions_zeta2(50)):>8.3f}'
      f' {acc_z_rdp.budget_cost(acc_z_rdp.compositions_zeta2(100)):>8.3f}'
      f'  {k_exhaust_z} hits')
print(f'{"Stitched (Prop 5)":<20}'
      f' {acc_s_rdp.budget_cost(acc_s_rdp.compositions_stitched(10)):>8.3f}'
      f' {acc_s_rdp.budget_cost(acc_s_rdp.compositions_stitched(50)):>8.3f}'
      f' {acc_s_rdp.budget_cost(acc_s_rdp.compositions_stitched(100)):>8.3f}'
      f'  {k_exhaust_s} hits')
print(f'{"Std IDP RDP":<20}'
      f' {10*rho:>8.3f}'
      f' {50*rho:>8.3f}'
      f' {100*rho:>8.3f}'
      f'  {k_exhaust_idp} hits')
print()
print(f'H1 half-lives: Zeta-2={hl_z}, Stitched={hl_s}, IDP={hl_i}')
print(f'c\' = {_C_PRIME:.6f}  (Proposition 5 stitched constant)')